# Per-frame latency + compute load — six detectors, CPU vs GPU

Two evals from one run of `run_latency_eval.py`, on a single simulated frame from the
**20 dB attenuation** capture, resampled to 20 / 100 / 250 / 500 MHz at the FFT geometry
the deployed pipeline auto-selects for each rate (`fft_sizing.py`, ported from
`fft_runtime_config.hpp`).

| detector | what it is | reference impl |
|---|---|---|
| `coherent_power` | deployed coherent power detector | torch FFT + CFAR + morphology |
| `cuda_dino` | zero-shot DINOv3 | frozen ViT-B/16 over 256×512 chunks |
| `3dB_power` | moving-average power baseline | torch FFT + scalar threshold |
| `blob_detection` | image-processing blobs baseline | torch conv edges + morphology |
| `yolo` | fine-tuned YOLO26-m | reused `yolo_infer.YoloDetector` |
| `dino_finetuned` | fine-tuned DINOv3 segmenter | reused `finetuned_infer.FinetunedDetector` |

**Eval 1 (compute load):** FLOPs/frame + peak GPU memory per detector per rate.  
**Eval 2 (latency):** average CPU and GPU per-frame latency per detector, with the
**real-time budget** (`samples_per_frame / sample_rate`) drawn as a labeled horizontal
dashed line.

All six run as device-switchable torch reference implementations so CPU and GPU are
measured on identical footing (the deployed `coherent_power`/`cuda_dino` are C++/CUDA with
no CPU path). See `README.md` for the fidelity caveats.

In [ ]:
# %load_ext autoreload
# %autoreload 2
import sys
from pathlib import Path

_HERE = Path.cwd()
if _HERE.name != 'latency_comp_eval':
    _HERE = _HERE / 'infocom_evals' / 'latency_comp_eval'
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

import numpy as np
from latency_results import LatencyResults
import plot_latency_results as pl

# Point at the run you want to review (base path; .npz/.json resolved automatically).
RESULTS = _HERE / 'saved_results' / 'latency_run'
results = LatencyResults.load(RESULTS)
print('detectors :', results.detectors())
print('rates MHz :', [r/1e6 for r in results.sample_rates()])
print('devices   :', sorted(set(np.asarray(results.cells['device']).astype(str))))
print('host/gpu  :', results.provenance.get('host'), '/', results.provenance.get('gpu'))

## Real-time budgets

The real-time deadline for one frame is `budget = num_ffts_per_batch × (fft_size / sample_rate)
= 512 × (FFT window time)`. The varying knob is the **FFT window / integration time** (`dwell =
fft_size/sample_rate = 1/bin_size`): a **longer FFT window → more budget**, and it's the one
unit that's both monotonic-with-budget and sample-rate-independent (so it's a single horizontal
line). Bin size is its reciprocal (coarser kHz = shorter window = *less* budget). The dashed
lines: 50 µs / 20 kHz → 25.6 ms, 20 µs / 50 kHz → 10.24 ms, 10 µs / 100 kHz → 5.12 ms, 5 µs /
200 kHz → 2.56 ms. (The deployed auto-FFT holds ~20–24 kHz/bin, so the per-rate frame duration
is ~constant ~21–26 ms.) See `README.md`.

In [ ]:
pl.print_summary(results)

## Eval 1 — compute load per frame (GFLOPs + peak GPU memory)

FLOPs = aten conv/matmul (torch `FlopCounterMode`) + analytic FFT flops (total per-frame
work, incl. the shared FFT front-end). Peak memory is the CUDA peak allocation to process
one frame. Both axes log — values span ~5 orders of magnitude across detectors.

In [ ]:
pl.fig_compute_load(results)

## Eval 2 — max real-time sample rate per detector

**The headline figure.** Detector-only latency vs sample rate (both log), one line per
detector on GPU. Latency *rises* with rate (more rate → proportionally more work per frame);
the real-time budget (frame duration) is ~constant ~21 ms because the FFT auto-scales with the
rate. **Where each detector's line crosses the budget = its max feasible sample rate** (marked,
and printed in the legend). Solid = measured (20–500 MS/s); dotted = power-law extrapolation
(`latency ∝ rate`) to locate crossovers beyond the tested range — treat the extrapolated max
rates (e.g. 3dB_power ~2.8 GS/s) as first-order estimates.

In [ ]:
pl.fig_max_rate(results)

### Per-rate detail — average latency, CPU vs GPU

Same latency data as clustered bars: one cluster per detector, a bar per sample rate × device
(CPU then GPU), with the FFT-window / bin-size real-time budgets as horizontal lines.

In [ ]:
pl.fig_latency_bars(results)

## Latency vs sample rate (CPU & GPU) with the real-time budget curve

In [ ]:
pl.fig_latency_vs_rate(results)

## Raw per-cell table

In [ ]:
cols = ['detector','device','sample_rate_hz','fft_size','n_reps','lat_median_ms','lat_p95_ms','gflops','peak_mem_mb','frame_budget_ms']
hdr = ['detector','dev','rate MHz','fft','n','med ms','p95 ms','GFLOPs','peak MB','budget ms']
print(' '.join(f'{h:>12}' for h in hdr))
order = np.lexsort((np.asarray(results.cells['sample_rate_hz'],float),
                    np.asarray(results.cells['device']).astype(str),
                    np.asarray(results.cells['detector']).astype(str)))
for i in order:
    row = []
    for c in cols:
        v = results.cells[c][i]
        if c == 'sample_rate_hz':
            row.append(f'{float(v)/1e6:>12.0f}')
        elif c in ('detector','device'):
            row.append(f'{str(v):>12}')
        else:
            row.append(f'{float(v):>12.2f}')
    print(' '.join(row))